In [2]:
VOCAB_SIZE    = 30522
EMBED_DIM     = 256
NUM_HEADS     = 4
NUM_LAYERS    = 2
FFN_DIM       = 512
MAX_LEN       = 128
PROJECTION    = 128
DROPOUT       = 0.3

In [3]:
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.position_embedding = nn.Embedding(MAX_LEN, EMBED_DIM)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS)


        self.projection = nn.Linear(EMBED_DIM, PROJECTION)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, input_ids, attention_mask):
        x = self.token_embedding(input_ids)

        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = x + self.position_embedding(positions)

        x = self.dropout(x)

        padding_mask = (attention_mask == 0)
        x = self.transformer(x, src_key_padding_mask=padding_mask)

        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1)
        x = self.projection(x)

        return F.normalize(x, dim=-1)

    def encode(self, texts):
        tokens = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(device)
        return self.forward(tokens["input_ids"], tokens["attention_mask"])

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [6]:
model = TransformerEncoder().to(device)

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/nlp-final/checkpoints/squad_v2/best_model.pt",
        map_location=device
    )
)

<All keys matched successfully>

In [10]:
model.eval()

TransformerEncoder(
  (token_embedding): Embedding(30522, 256, padding_idx=0)
  (position_embedding): Embedding(128, 256)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (projection): Linear(in_features=256, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [18]:
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [22]:
import json

test_data  = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/squad/test.jsonl")

In [23]:
print(f"test pairs: {len(test_data)}")
print(test_data[0])

test pairs: 7880
{'query': 'what is the largest populated city in arizona?', 'document': 'tucson (/ˈtuːsɒn/ /tuːˈsɒn/) is a city and the county seat of pima county, arizona, united states, and home to the university of arizona. the 2010 united states census put the population at 520,116, while the 2013 estimated population of the entire tucson metropolitan statistical area (msa) was 996,544. the tucson msa forms part of the larger tucson-nogales combined statistical area (csa), with a total population of 980,263 as of the 2010 census. tucson is the second-largest populated city in arizona behind phoenix, both of which anchor the arizona sun corridor. the city is located 108 miles (174 km) southeast of phoenix and 60 mi (97 km) north of the u.s.-mexico border. tucson is the 33rd largest city and the 59th largest metropolitan area in the united states. roughly 150 tucson companies are involved in the design and manufacture of optics and optoelectronics systems, earning tucson the nicknam

In [24]:
seen = set()
docs = []
doc_id_map = {}

for item in test_data:
    doc_text = item["document"]
    if doc_text not in seen:
        seen.add(doc_text)
        doc_id_map[doc_text] = len(docs)
        docs.append(doc_text)

print(f"unique docs: {len(docs)}")

unique docs: 1721


In [26]:
import numpy as np

@torch.no_grad()
def encode_texts(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch)
        embeddings.append(emb.cpu().numpy())

    return np.concatenate(embeddings, axis=0)

doc_embeddings = encode_texts(docs)
print(doc_embeddings.shape)

(1721, 128)


In [28]:
queries = [item["query"] for item in test_data]
correct_doc_idx = [doc_id_map[item["document"]] for item in test_data]

query_embeddings = encode_texts(queries)
print(query_embeddings.shape)

(7880, 128)


In [29]:
sims = query_embeddings @ doc_embeddings.T

K = 5
retrieved = np.argsort(-sims, axis=1)[:, :K]

In [30]:
def compute_metrics(retrieved, correct_idx, k_values=[1, 5]):
    n = len(correct_idx)
    results = {}
    for k in k_values:
        hits = sum(1 for i in range(n) if correct_idx[i] in retrieved[i][:k])
        results[f"recall@{k}"] = hits / n

    rr_sum = 0
    for i in range(n):
        if correct_idx[i] in retrieved[i]:
            rank = list(retrieved[i]).index(correct_idx[i]) + 1
            rr_sum += 1 / rank
    results["mrr"] = rr_sum / n
    return results

metrics = compute_metrics(retrieved, correct_doc_idx)
print("Model:", metrics)

Model: {'recall@1': 0.2445431472081218, 'recall@5': 0.4484771573604061, 'mrr': 0.31974619289340095}


In [31]:
!pip install rank_bm25 -q
from rank_bm25 import BM25Okapi

tokenized_docs = [doc.lower().split() for doc in docs]
bm25 = BM25Okapi(tokenized_docs)

bm25_retrieved = []
for item in test_data:
    query_tokens = item["query"].lower().split()
    scores_bm25 = bm25.get_scores(query_tokens)
    top_k_idx = np.argsort(scores_bm25)[::-1][:K]
    bm25_retrieved.append(top_k_idx)

bm25_retrieved = np.array(bm25_retrieved)
bm25_metrics = compute_metrics(bm25_retrieved, correct_doc_idx)
print("BM25:", bm25_metrics)
print("Model:", metrics)

BM25: {'recall@1': 0.6232233502538072, 'recall@5': 0.7906091370558376, 'mrr': 0.6886019458544808}
Model: {'recall@1': 0.2445431472081218, 'recall@5': 0.4484771573604061, 'mrr': 0.31974619289340095}


In [32]:
def search(query, k=5):
    q_emb = encode_texts([query])
    sims = q_emb @ doc_embeddings.T
    top_k = np.argsort(-sims[0])[:k]
    for rank, idx in enumerate(top_k, 1):
        print(f"{rank}. (score={sims[0][idx]:.3f}) {docs[idx][:200]}")

search("what is the largest populated city in arizona?")

1. (score=0.629) tucson is located 118 mi (190 km) southeast of phoenix and 60 mi (97 km) north of the united states - mexico border. the 2010 united states census puts the city's population at 520,116 with a metropol
2. (score=0.567) in recent years the city government, under mayor manny diaz, has taken an ambitious stance in support of bicycling in miami for both recreation and commuting. every month, the city hosts "bike miami",
3. (score=0.562) city transportation in strasbourg includes the futurist-looking strasbourg tramway that opened in 1994 and is operated by the regional transit company compagnie des transports strasbourgeois (cts), co
4. (score=0.555) tucson (/ˈtuːsɒn/ /tuːˈsɒn/) is a city and the county seat of pima county, arizona, united states, and home to the university of arizona. the 2010 united states census put the population at 520,116, w
5. (score=0.553) the expansive area northwest of the city limits is diverse, ranging from the rural communities of catalina and 

In [36]:
search("who wrote the declaration of independence?")
print("--------------")
search("when did world war 2 end?")
print("--------------")
search("what is the capital of france?")

1. (score=0.596) in the 1950s and the 1960s, paris became one front of the algerian war for independence; in august 1961, the pro-independence fln targeted and killed 11 paris policemen, leading to the imposition of a
2. (score=0.519) william pitt, who entered the cabinet in 1756, had a grand vision for the war that made it entirely different from previous wars with france. as prime minister pitt committed britain to a grand strate
3. (score=0.510) in 1977, hayek was critical of the lib-lab pact, in which the british liberal party agreed to keep the british labour government in office. writing to the times, hayek said, "may one who has devoted a
4. (score=0.509) in france the cathars grew to represent a popular mass movement and the belief was spreading to other areas. the cathar crusade was initiated by the roman catholic church to eliminate the cathar heres
5. (score=0.509) the british government was close to bankruptcy, and britain now faced the delicate task of pacifying its new fr